# Topical Lectures, Controls
## Flight data, part 1: the flight plan

Andreas Freise, 01.11.2026


This notebook needs no drone and no radio. It is meant to help the pilot choose the loop gains. The pilot needs two things: before the first tuning flight we should estimate the gains worth flying, in which order, with the predicted rise time and overshoot for each. Before every further flight, we should provide the prediction from a model in this notebook for the drone gains the pilot is about to try. From the pilot we need, as soon as they are recorded, the sign of the gains from the hand test and `TAU`, `DELAY` and the two noise bounds from the identification flight. Until we have them, the typical values given below let us start.

The drone model is written out below (fifteen lines) so that we can see what it assumes and change it. Its flights can also be saved, for the comparison with the real flights afterwards.


In [ ]:
# Importing packages
import numpy as np
import matplotlib.pyplot as plt
import module
import crazyflie

%matplotlib inline

## The parameters of the drone

The model needs the sign of the gains, the response time `TAU` and the dead time `DELAY` of the velocity loop, and the two noise bounds. They are measured on the drone laptop, in the hand test and in the identification flight. Until your group has them, use typical values of the course drones: a negative sign, a response time of 0.13 s, a dead time of 0.20 s, and bounds of about 25 1/s for `|Kp|` and 3 for `|Kd|`. Replace them by your own as soon as they are recorded; the predictions are only as good as these numbers.


In [ ]:
KP_SIGN = None   # +1 or -1, the sign of the gains
TAU = None       # s, the response time of the velocity loop
DELAY = None     # s, the dead time
KP_MAX = None    # 1/s, the bound on |Kp| from the noise
KD_MAX = None    # the bound on |Kd| from the noise

STEP = 0.5       # m, the step of the target, as in the flights
v_max = 0.3      # m/s, the limit on the feedback, as in the flights
dt = 0.02        # s, the loop period, as in the flights

## The model

The drone is reduced to its velocity loop: the measured velocity follows the feedback as a first order response with the time constant `TAU`, and the feedback acts `DELAY` seconds after our loop computed it. The position is the integral of the velocity. Our controller is the `PIDController` of the drone notebooks; one axis is enough here. The loop runs at the same 50 Hz as in flight, holds the start for 2 s, steps the target by `STEP` and holds for 4 s, and the feedback is clamped at `v_max`. The second cell reads a step response the way the drone notebooks do, rise time, overshoot and the error in the last two seconds, and the third is the controller of the drone notebooks, needed for the saved model flights further down.


In [ ]:
def model_flight(Kp, Kd, Ki, tau=None, delay=None, step=None, hold_s=2.0, step_s=4.0):
    """One step of the target on the model drone. Returns t (seconds from the step), x, v and the feedback."""
    tau, delay, step = (TAU if tau is None else tau), (DELAY if delay is None else delay), (STEP if step is None else step)
    n, n_delay = int(round((hold_s + step_s) / dt)), int(round(delay / dt))
    t = np.arange(n) * dt - hold_s
    target = np.where(t >= 0, step, 0.0)
    x, v, fb = np.zeros(n), np.zeros(n), np.zeros(n)
    ctrl = module.PIDController(Kp=Kp, Ki=Ki, Kd=Kd, deltaT=dt)
    for i in range(1, n):
        fb[i - 1] = float(np.clip(ctrl(x[i - 1], v[i - 1], target[i - 1]), -v_max, v_max))   # what our loop sends
        acting = fb[i - 1 - n_delay] if i - 1 - n_delay >= 0 else 0.0                        # what reaches the drone now
        v[i] = v[i - 1] + (acting - v[i - 1]) * dt / tau                                      # the velocity loop follows
        x[i] = x[i - 1] + v[i] * dt                                                           # the position is its integral
    return t, x, v, fb


In [ ]:
def step_numbers(t, x, step, fb=None, v_max=None):
    """Read one step response: t in seconds from the step, x in metres from the start, step the change of the target."""
    after = t >= 0
    t, x, step = t[after], x[after] * np.sign(step), abs(step)
    hit = x > 0.9 * step
    rise = float(t[np.argmax(hit)]) if hit.any() else None          # first time beyond 90 % of the step
    overshoot = float(max(0.0, x.max() - step))                     # the largest excursion beyond the target
    tail = t >= t[-1] - 2.0                                         # the last 2 s of the step phase
    err = step - x[tail]
    numbers = {'rise_s': rise, 'overshoot_m': overshoot, 'max_error_m': float(np.abs(err).max()),
               'mean_error_m': float(err.mean()), 'settled': bool(np.abs(err).max() < 0.15)}
    if fb is not None and v_max is not None:
        numbers['clamp_fraction'] = float((np.abs(fb[after]) >= v_max - 1e-3).mean())   # share of the step at the limit
    return numbers

def print_numbers(*columns):
    """A small table of step numbers: print_numbers(('model', pred), ('flight', meas))."""
    show = lambda v: '--' if v is None else ('yes' if v is True else 'no' if v is False else f'{v:.2f}')
    print(f"{'':26s}" + ''.join(f'{name:>12s}' for name, _ in columns))
    for key, text in (('rise_s', 'rise to 90 % [s]'), ('overshoot_m', 'overshoot [m]'), ('max_error_m', 'max error, last 2 s [m]'),
                      ('mean_error_m', 'mean error, last 2 s [m]'), ('clamp_fraction', 'share at v_max'), ('settled', 'settled (< 0.15 m)')):
        if any(key in n for _, n in columns):
            print(f'{text:26s}' + ''.join(f'{show(n.get(key)):>12s}' for _, n in columns))


In [ ]:
def make_controller(Kp, Kd, Ki):
    """One PID per axis with these gains. Returns the controller that fly() calls and the reset of its integrators."""
    x_ctrl = module.PIDController(Kp=Kp, Ki=Ki, Kd=Kd, deltaT=dt)
    y_ctrl = module.PIDController(Kp=Kp, Ki=Ki, Kd=Kd, deltaT=dt)

    def position_controller(x, y, vx, vy, x_target, y_target):
        """Position and velocity in, feedback (a velocity) out, clipped to v_max."""
        vx_fb = x_ctrl(x, vx, x_target)
        vy_fb = y_ctrl(y, vy, y_target)
        return float(np.clip(vx_fb, -v_max, v_max)), float(np.clip(vy_fb, -v_max, v_max))

    def reset_controllers():
        """Zero the integrators, called before every flight."""
        x_ctrl.reset()
        y_ctrl.reset()

    return position_controller, reset_controllers


## The flight plan

Before the first tuning flight. The cell flies the model for a series of `Kp` with `Kd = Ki = 0`, then for `Kd` at a large `Kp`, then for `Ki` at a moderate one, and writes rise time and overshoot into the legends. The plan comes out of these three panels. From the first, the `Kp` for which the model arrives in two to three seconds and overshoots less than 5 cm: that is the first real gain for the pilot, after the deliberately small one. Then the `Kp` series the pilot should fly, doubling from there, up to where the model overshoots more than 10 cm; the real drone reaches that point earlier. From the second panel, the `Kd` that takes the overshoot away at that `Kp`, and from the third what `Ki` does to the step. Change the series in the cell as you need. Record the plan below, in the order to fly, with the predicted numbers, and give it to the pilot.


In [ ]:
series = [('Kp', [(KP_SIGN * k, 0.0, 0.0) for k in (0.5, 1.0, 2.0, 4.0, 8.0)]),
          ('Kd at Kp = 4', [(KP_SIGN * 4.0, Kd, 0.0) for Kd in (0.3, 0.0, -0.3, -0.6)]),
          ('Ki at Kp = 1', [(KP_SIGN * 1.0, 0.0, KP_SIGN * k) for k in (0.0, 0.1, 0.3)])]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
for ax, (title, gains) in zip(axes, series):
    for Kp_, Kd_, Ki_ in gains:
        t_m, x_m, _, _ = model_flight(Kp_, Kd_, Ki_)
        n = step_numbers(t_m, x_m, STEP)
        rise = f"{n['rise_s']:.1f} s" if n['rise_s'] is not None else 'not reached'
        ax.plot(t_m, x_m, label=f"Kp {Kp_:+.1f}, Kd {Kd_:+.1f}, Ki {Ki_:+.1f}: rise {rise}, overshoot {100 * n['overshoot_m']:.0f} cm")
    ax.axhline(STEP, color='k', lw=0.5, ls='--')
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.set_xlim(-1, 8)
    ax.set_xlabel('time since step [s]')
    ax.set_title(title, fontsize=10)
    ax.legend(loc='lower right', fontsize=7)
    ax.grid()
axes[0].set_ylabel('position from start [m]')
plt.tight_layout()


In [ ]:
# The flight plan for the pilot, in the order to fly, with the predicted numbers from the legends above:
PLAN = [
    dict(Kp=None, Kd=0.0, Ki=0.0, predicted_rise_s=None, predicted_overshoot_m=None, note='first real gain'),
]


## Predicting the next flight

Before each flight. Enter the gains the pilot is about to fly and run the cell: it flies the model for them, prints the numbers and draws the curve, with and without the dead time. The pilot writes the predicted rise time into the record of the flight and compares after landing. Where the real flight departs from the model, something the model does not contain is at work: the attitude loop, the air in the room, the battery.


In [ ]:
Kp = None     # 1/s, sign included
Kd = None
Ki = None
if None in (Kp, Kd, Ki):
    raise ValueError('enter the three gains the pilot is about to fly')

t_m, x_m, v_m, fb_m = model_flight(Kp, Kd, Ki)
t_f, x_f, _, fb_f = model_flight(Kp, Kd, Ki, delay=0.0)
print_numbers(('with delay', step_numbers(t_m, x_m, STEP, fb_m, v_max)), ('no delay', step_numbers(t_f, x_f, STEP, fb_f, v_max)))
fig, ax = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
ax[0].plot(t_m, x_m, label=f'model, Kp = {Kp:+.2f}, Kd = {Kd:+.2f}, Ki = {Ki:+.2f}')
ax[0].plot(t_f, x_f, ls=':', label='the same without the dead time')
ax[0].axhspan(STEP - 0.15, STEP + 0.15, color='0.92', zorder=0)
ax[0].axhline(STEP, color='k', lw=0.5, ls='--')
ax[0].axhline(0, color='k', lw=0.5, ls='--')
ax[0].set_ylabel('position from start [m]')
ax[0].legend(loc='lower right')
ax[0].grid()
ax[1].plot(t_m, fb_m, label='feedback')
ax[1].plot(t_m, v_m, alpha=0.6, label='velocity of the model')
ax[1].axhline(v_max, color='k', lw=0.5, ls='--')
ax[1].axhline(-v_max, color='k', lw=0.5, ls='--')
ax[1].set_xlim(-1, 8)
ax[1].set_xlabel('time since step [s]')
ax[1].set_ylabel('velocity [m/s]')
ax[1].legend(loc='upper right')
ax[1].grid()
plt.tight_layout()


## Model flights saved like real ones

`crazyflie.fly_model` runs the same model inside the module, with the noise and the drift of the estimate added, through the same code path as a real flight, and `crazyflie.save` writes the same files, with `model` in place of the drone label. Saved model flights appear next to the real ones in the comparison notebook. Use the labels of the pilot, so that a real flight and its model twin carry the same name.


In [ ]:
RUN_LABEL = 'kp1'       # the same label as the pilot uses for these gains

position_controller, reset_controllers = make_controller(Kp, Kd, Ki)
run = crazyflie.fly_model(position_controller, axes='x', step=STEP, v_max=v_max, reset=reset_controllers,
                          tau=TAU, delay=DELAY, info=dict(Kp=Kp, Kd=Kd, Ki=Ki, tau=TAU, delay=DELAY))
name = crazyflie.save(run, RUN_LABEL)
